# Team runbook — training curves (+ optional bounded QPU fine-tune)

**What this notebook produces:** `data/loss_curves.npz` — the recorded training
histories (QNN + parameter-matched MLP) that the judge notebook's
*Training dynamics* figure loads. Run it top to bottom, then commit the
generated `.npz` (or send it to Karla) and the figure lights up.

**Run it from the `qnn_tc/` folder with the project venv**
(`.venv/bin/jupyter lab` or select the `.venv` kernel), so `qnn.py` and
`prep_data.py` import correctly.

| Profile | What it is | Wall time (laptop) |
|---|---|---|
| `fast` (default) | 300 rows, maxiter 60, seeds 7/17/27 — the exact multiseed-study protocol | ~30–45 min |
| `full` | 800 rows, maxiter 150, seed 7 — the champion-training protocol | overnight |
| `bigger_data` | regenerate a larger prepared set first (see below) | days+ |


## Read this first: two honest cost facts

**1 · Why we do NOT train on the QPU.** Training needs gradients. Parameter-shift
gradients cost **2 circuits per parameter per training row per optimizer step**:
with 32 parameters and 300 rows that is ~19,200 circuit executions *per step*,
and ~60 steps ≈ **1.15 million QPU circuit executions** for one training run —
days of queue time and many times our entire allowance. This is why the standard
(and challenge-prescribed) workflow is **hybrid**: train on the classical
simulator of the exact circuit, then spend scarce QPU time on inference and
validation — where we already showed hardware matches simulation
(22.89 K vs 23.00 K). A *bounded* hardware-in-the-loop fine-tune demo IS
feasible and is provided at the bottom of this notebook (~9 QPU-min, gated off
by default).

**2 · Why 300 rows is not a compromise (the 21,263-row question).** The dataset
has 21,263 materials, but our 32-parameter QNN is **capacity-limited, not
data-limited** — we measured this: trained at the full budget (800 rows,
maxiter 150) the champion reaches **19.8 K**; trained at the sweep budget
(300 rows, maxiter 60) it reaches **19.84 ± 0.03 K** across three seeds. Same
number. A 32-parameter model underfits long before data runs out, so more rows
buy nothing measurable while simulation cost grows linearly with rows
(21k rows ≈ 70× slower ≈ weeks per run). If you still want to try:
`prep_data.prepare(n_features=4, n_train=5000, ...)` regenerates a larger
prepared set — keep `n_test` and the seed unchanged so results stay comparable,
and expect the same RMSE.

In [ ]:
# ---------------- configuration ----------------
PROFILE = "fast"          # "fast" | "full"  (see the table above)

PROFILES = {
    "fast": dict(n_train=300, maxiter=60,  seeds=(7, 17, 27)),
    "full": dict(n_train=800, maxiter=150, seeds=(7,)),
}

RUN_QPU_FINETUNE = False  # read the section at the bottom before flipping this
QPU_BACKEND = "ibm_fez"
QPU_ROWS = 32             # training subset for the bounded fine-tune demo
QPU_ITERS = 10            # SPSA iterations (2 QPU evaluations each)
QPU_SHOTS = 2048

cfg = PROFILES[PROFILE]
print(f"profile {PROFILE}: {cfg}")

In [ ]:
import time
import warnings

import numpy as np
from qiskit_machine_learning.algorithms import NeuralNetworkRegressor
from qiskit_machine_learning.optimizers import L_BFGS_B
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor

from prep_data import inverse_target
from qnn import make_qnn

warnings.filterwarnings("ignore", category=ConvergenceWarning)

MLP_EPOCHS = 400
MAX_RMSE_CHECKPOINTS = 40


def rmse_kelvin_from_pred(y_true_scaled, y_pred_scaled, d):
    t = inverse_target(np.asarray(y_true_scaled).ravel(), d["y_min"], d["y_max"])
    p = inverse_target(np.clip(np.asarray(y_pred_scaled).ravel(), -1, 1),
                       d["y_min"], d["y_max"])
    return float(np.sqrt(np.mean((t - p) ** 2)))


def train_qnn_with_history(d, seed, n_train, maxiter):
    qnn, _ = make_qnn(4, 2, d["X_train"].shape[1])
    objs, weight_log = [], []

    def callback(weights, obj):
        objs.append(float(obj))
        weight_log.append(np.array(weights, copy=True))

    rng = np.random.default_rng(seed)
    reg = NeuralNetworkRegressor(
        neural_network=qnn, loss="squared_error",
        optimizer=L_BFGS_B(maxiter=maxiter),
        initial_point=rng.uniform(-0.3, 0.3, qnn.num_weights),
        callback=callback)
    t0 = time.time()
    reg.fit(d["X_train"][:n_train], d["y_train"][:n_train])
    fit_s = time.time() - t0

    idx = np.unique(np.linspace(0, len(weight_log) - 1,
                                min(MAX_RMSE_CHECKPOINTS, len(weight_log)),
                                dtype=int))
    rmses = [rmse_kelvin_from_pred(
        d["y_test"], np.asarray(qnn.forward(d["X_test"], weight_log[i])).ravel(), d)
        for i in idx]
    final = rmse_kelvin_from_pred(
        d["y_test"], np.asarray(qnn.forward(d["X_test"], reg.weights)).ravel(), d)
    print(f"QNN seed {seed}: {len(objs)} evals, fit {fit_s:.0f}s, "
          f"final test RMSE {final:.2f} K")
    return np.array(objs), idx, np.array(rmses), final


def train_mlp_with_history(d, seed, n_train):
    mlp = MLPRegressor(hidden_layer_sizes=(5,), max_iter=1, warm_start=True,
                       random_state=seed)
    losses, rmses = [], []
    for _ in range(MLP_EPOCHS):
        mlp.fit(d["X_train"][:n_train], d["y_train"][:n_train])
        losses.append(float(mlp.loss_))
        rmses.append(rmse_kelvin_from_pred(d["y_test"], mlp.predict(d["X_test"]), d))
    print(f"MLP seed {seed}: {MLP_EPOCHS} epochs, final test RMSE {rmses[-1]:.2f} K")
    return np.array(losses), np.array(rmses)

In [ ]:
d = dict(np.load("data/prepared_4d_pls.npz"))
out = {"seeds": np.array(cfg["seeds"]), "n_train": cfg["n_train"],
       "maxiter": cfg["maxiter"], "mlp_epochs": MLP_EPOCHS}
for seed in cfg["seeds"]:
    objs, idx, rmses, final = train_qnn_with_history(
        d, seed, cfg["n_train"], cfg["maxiter"])
    out[f"qnn_obj_s{seed}"] = objs
    out[f"qnn_ckpt_idx_s{seed}"] = idx
    out[f"qnn_rmse_s{seed}"] = rmses
    out[f"qnn_final_s{seed}"] = final
    losses, mlp_rmses = train_mlp_with_history(d, seed, cfg["n_train"])
    out[f"mlp_loss_s{seed}"] = losses
    out[f"mlp_rmse_s{seed}"] = mlp_rmses

np.savez("data/loss_curves.npz", **out)
print("\nsaved data/loss_curves.npz — commit this file (or send it to Karla).")

In [ ]:
# quick visual check of what you just produced
import matplotlib.pyplot as plt

lc = np.load("data/loss_curves.npz")
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for s in lc["seeds"]:
    axes[0].plot(lc[f"qnn_ckpt_idx_s{s}"], lc[f"qnn_rmse_s{s}"], lw=1.8, label=f"QNN seed {s}")
    mrm = lc[f"mlp_rmse_s{s}"]
    axes[1].plot(np.arange(1, len(mrm) + 1), mrm, lw=1.8, label=f"MLP seed {s}")
axes[0].set(xlabel="L-BFGS-B objective evaluation", ylabel="test RMSE (K)", title="QNN")
axes[1].set(xlabel="training epoch", title="param-matched MLP")
for ax in axes:
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylim(top=min(46, axes[0].get_ylim()[1]))
plt.tight_layout(); plt.show()

## Optional: bounded QPU fine-tune demo (~9 QPU-minutes)

This is **not** full training on hardware (see the cost fact at the top). It is
a deliberately bounded demonstration of hardware-in-the-loop optimization:

- start from the **shipped champion weights** (already trained in simulation),
- SPSA optimizer with **fixed** learning rate and perturbation (fixing them
  skips SPSA's auto-calibration, which would otherwise burn ~50 extra QPU
  evaluations),
- 32 training rows × 10 iterations × 2 evaluations/iteration ≈ **~672 circuit
  executions ≈ 9 QPU-minutes** at 2048 shots with TREX,
- runs inside one Session on the `Quantum-Hackathon-Cern` premium instance.

What it honestly demonstrates: the optimizer can take real steps against
hardware expectation values, and the loss it sees on the QPU tracks the
simulator loss — consistent with our flat noise ladder. Do not expect the RMSE
to improve on an already-trained model; say so if asked.

Flip `RUN_QPU_FINETUNE = True` in the config cell, re-run it, then run the next
cell. IBM credentials must already be saved (they are, on Karla's account).

In [ ]:
if RUN_QPU_FINETUNE:
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService, Session
    from qiskit_machine_learning.optimizers import SPSA

    d = dict(np.load("data/prepared_4d_pls.npz"))
    Xf = d["X_train"][:QPU_ROWS]
    yf = d["y_train"][:QPU_ROWS]
    w0 = np.load("data/trained_weights_q4_l2_pls.npy")

    service = QiskitRuntimeService()
    backend = service.backend(QPU_BACKEND)
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

    history = []
    with Session(backend=backend) as session:
        est = EstimatorV2(mode=session)
        est.options.default_shots = QPU_SHOTS
        est.options.resilience_level = 1           # TREX readout mitigation
        qnn_hw, _ = make_qnn(4, 2, 4, estimator=est, pass_manager=pm)

        def objective(w):
            # EstimatorQNN.forward handles parameter ordering correctly —
            # never bind bare arrays to PUBs yourself (see the binding-bug
            # section of the judge notebook).
            pred = np.asarray(qnn_hw.forward(Xf, w)).ravel()
            val = float(np.mean((pred - yf) ** 2))
            history.append({"obj": val, "weights": np.array(w)})
            print(f"QPU eval {len(history):2d}: scaled MSE {val:.5f}")
            return val

        spsa = SPSA(maxiter=QPU_ITERS, learning_rate=0.02, perturbation=0.05)
        result = spsa.minimize(objective, x0=np.array(w0))

    objs = np.array([h["obj"] for h in history])
    weights_log = np.stack([h["weights"] for h in history])
    np.savez("data/qpu_finetune.npz", objs=objs, weights=weights_log,
             final_weights=np.array(result.x), backend=QPU_BACKEND,
             rows=QPU_ROWS, iters=QPU_ITERS, shots=QPU_SHOTS)
    print(f"\nsaved data/qpu_finetune.npz — {len(history)} QPU evaluations.")
else:
    print("RUN_QPU_FINETUNE is False — skipped (this is the safe default).")

## When you are done

1. Check the figure above looks sane (QNN curves overlapping near ~20 K,
   MLP curves spread out).
2. `git add data/loss_curves.npz` (and `data/qpu_finetune.npz` if you ran the
   QPU cell), commit, push — or send the files to Karla.
3. The judge notebook (`BasQ_QNN_Tc.ipynb`) picks up `loss_curves.npz`
   automatically in its *Training dynamics* cell. Nothing else to wire up.